In [1]:
import random

# Συναρτήσεις από το Εργαστήριο 3
def mod_exp(base, exp, mod):
    result = 1
    base = base % mod
    while exp > 0:
        if exp % 2 == 1:
            result = (result * base) % mod
        base = (base * base) % mod
        exp //= 2
    return result

def extended_gcd(a, b):
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd, x, y

def mod_inverse(a, m):
    gcd, x, y = extended_gcd(a, m)
    if gcd != 1:
        raise Exception('Δεν υπάρχει Modulo Inverse')
    return x % m


# Υλοποίηση Αλγορίθμου Paillier

def paillier_keygen(p, q):
    n = p * q
    n_sq = n * n
    lmbda = (p - 1) * (q - 1)  # Απλοποιημένο (αντί για Ε.Κ.Π.)
    g = n + 1
    mu = mod_inverse(lmbda, n)

    # Επιστρέφουμε: (public_key), (private_key)
    return (n, g, n_sq), (lmbda, mu)

def paillier_encrypt(m, pub_key):
    n, g, n_sq = pub_key

    # Επιλογή τυχαίου r
    r = random.randint(1, n - 1)

    # c = (g^m * r^n) mod n^2
    term1 = mod_exp(g, m, n_sq)
    term2 = mod_exp(r, n, n_sq)

    c = (term1 * term2) % n_sq
    return c

def paillier_decrypt(c, pub_key, priv_key):
    n, g, n_sq = pub_key
    lmbda, mu = priv_key

    # u = c^λ mod n^2
    u = mod_exp(c, lmbda, n_sq)

    # L(u) = (u - 1) // n
    l = (u - 1) // n

    # m = (L(u) * μ) mod n
    m = (l * mu) % n
    return m


# Εκτέλεση
print("=== Paillier ===\n")

# Επιλέγουμε δύο μικρούς πρώτους αριθμούς για το εργαστήριο
p = 17
q = 19
print(f"1. Δημιουργία κλειδιών με p={p}, q={q}")
pub_key, priv_key = paillier_keygen(p, q)
n, g, n_sq = pub_key

# Τα αρχικά μας μηνύματα
m1 = 5
m2 = 10
print(f"\n2. Έχουμε τα μηνύματα: m1 = {m1}, m2 = {m2}")

# Κρυπτογράφηση
c1 = paillier_encrypt(m1, pub_key)
c2 = paillier_encrypt(m2, pub_key)
print(f"   Κρυπτογραφημένο m1 (c1): {c1}")
print(f"   Κρυπτογραφημένο m2 (c2): {c2}")

# ΟΜΟΜΟΡΦΙΚΗ ΙΔΙΟΤΗΤΑ
print("\n3. Κάνουμε ΠΟΛΛΑΠΛΑΣΙΑΣΜΟ στα κρυπτογραφημένα δεδομένα (στο Cloud)...")
# Πολλαπλασιάζουμε τα ciphertexts και παίρνουμε modulo n^2
c_sum = (c1 * c2) % n_sq
print(f"   Το νέο ciphertext (c_sum) είναι: {c_sum}")

# ΑΠΟΚΡΥΠΤΟΓΡΑΦΗΣΗ
print("\n4. Αποκρυπτογραφούμε το c_sum (τοπικά)...")
decrypted_sum = paillier_decrypt(c_sum, pub_key, priv_key)

print(f"   Το αποκρυπτογραφημένο αποτέλεσμα είναι: {decrypted_sum}")

if decrypted_sum == (m1 + m2):
    print("\nΕΠΙΤΥΧΙΑ! Ο πολλαπλασιασμός των κρυπτογραφημένων μετατράπηκε σε πρόσθεση!")
else:
    print("\nΚάτι πήγε στραβά...")

=== Paillier ===

1. Δημιουργία κλειδιών με p=17, q=19

2. Έχουμε τα μηνύματα: m1 = 5, m2 = 10
   Κρυπτογραφημένο m1 (c1): 35522
   Κρυπτογραφημένο m2 (c2): 53630

3. Κάνουμε ΠΟΛΛΑΠΛΑΣΙΑΣΜΟ στα κρυπτογραφημένα δεδομένα (στο Cloud)...
   Το νέο ciphertext (c_sum) είναι: 101649

4. Αποκρυπτογραφούμε το c_sum (τοπικά)...
   Το αποκρυπτογραφημένο αποτέλεσμα είναι: 15

ΕΠΙΤΥΧΙΑ! Ο πολλαπλασιασμός των κρυπτογραφημένων μετατράπηκε σε πρόσθεση!


In [2]:
!pip install phe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.2 MB/s eta 0:00:00


In [3]:
from phe import paillier

print("=== Σενάριο: Ασφαλής Υπολογισμός Μέσου Όρου (Paillier) ===\n")

# 1. Παραγωγή Κλειδιών (Το κάνει ο τελικός αποδέκτης των πληροφοριών)
print("1. Παραγωγή κλειδιών Paillier...")
public_key, private_key = paillier.generate_paillier_keypair()

# 2. Η πλευρά των Υπαλλήλων (Κρυπτογράφηση)
salary_alice = 1200
salary_bob = 1500

print(f"2. Κρυπτογράφηση μισθών (Αλίκη: {salary_alice}, Μπομπ: {salary_bob})")
enc_alice = public_key.encrypt(salary_alice)
enc_bob = public_key.encrypt(salary_bob)

# Ας δούμε πώς φαίνεται ο κρυπτογραφημένος μισθός!
print(f"   Κρυπτογραφημένος Μισθός Αλίκης (Ciphertext): {str(enc_alice.ciphertext())}...")

# 3. Το Cloud / Λογιστήριο (Υπολογισμός σε κρυπτογραφημένα δεδομένα)
# Το λογιστήριο ΔΕΝ έχει το private_key. Κάνει την πρόσθεση στα "τυφλά"!
print("\n3. Το Cloud προσθέτει τα κρυπτογραφημένα δεδομένα...")
enc_sum = enc_alice + enc_bob

# (Ομομορφική ιδιότητα: Μπορούμε να προσθέσουμε έναν κανονικό (scalar) αριθμό!)
# Π.χ., δίνουμε bonus 100 ευρώ σε όλους συνολικά
enc_sum_with_bonus = enc_sum + 100
print("   Ο υπολογισμός ολοκληρώθηκε στο Cloud!")

# 4. Επιστροφή αποτελέσματος και Αποκρυπτογράφηση
print("\n4. Αποκρυπτογράφηση αποτελέσματος (Μόνο ο κάτοχος του Private Key)...")
decrypted_sum = private_key.decrypt(enc_sum_with_bonus)

# Υπολογισμός Μέσου Όρου
# (Η διαίρεση με το 2 γίνεται ΜΕΤΑ την αποκρυπτογράφηση, στο τελικό νούμερο)
average_salary = decrypted_sum / 2

print(f"   Συνολικό Άθροισμα (με το Bonus): {decrypted_sum}")
print(f"   Μέσος Όρος (ανά υπάλληλο):       {average_salary}")

=== Σενάριο: Ασφαλής Υπολογισμός Μέσου Όρου (Paillier) ===

1. Παραγωγή κλειδιών Paillier...
2. Κρυπτογράφηση μισθών (Αλίκη: 1200, Μπομπ: 1500)
   Κρυπτογραφημένος Μισθός Αλίκης (Ciphertext): 6939652244084696703242380646020741232777123018559941152635640146973951036002498274738472144456468476714048213656239600465232518912693053863583176363293318840699540448002232740048692293876102731554180879862708979673859422672107029551847893141491444843479355247056100657111935414397188149025738484595161277998731038609451905098615215379707684446810796027356166769127668061294860071389855012282144673578060685963537342642438928024805147743757458070840133211121514240086246950948756596862093873832058983197475513142557464697731768529829479157641468374323180681234897730169543754094940079192439721137769159046830883308371445682330561914970820837987962817263315504183650790835409051203029500488531880331747777893552223819979648281048562021425198941698820836176109586314571060093605150003634733099113001741994528130

In [4]:
print("\n=== Μέρος 4: Τα Όρια του Paillier ===")
print("Τι γίνεται αν προσπαθήσουμε να ΠΟΛΛΑΠΛΑΣΙΑΣΟΥΜΕ δύο κρυπτογραφημένα νούμερα;")

try:
    # Προσπαθούμε να πολλαπλασιάσουμε τον κρυπτογραφημένο μισθό της Αλίκης με του Μπομπ
    enc_multiplication = enc_alice * enc_bob
    print("Επιτυχία!")
except NotImplementedError as e:
    print(f"ΣΦΑΛΜΑ! Ο αλγόριθμος το απαγορεύει: {e}")


=== Μέρος 4: Τα Όρια του Paillier ===
Τι γίνεται αν προσπαθήσουμε να ΠΟΛΛΑΠΛΑΣΙΑΣΟΥΜΕ δύο κρυπτογραφημένα νούμερα;
ΣΦΑΛΜΑ! Ο αλγόριθμος το απαγορεύει: Good luck with that...


In [5]:
!pip install tenseal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 39.2 MB/s eta 0:00:00


In [6]:
import tenseal as ts

print("=== CKKS ===")

# 1. Δημιουργία Context (Οι παράμετροι του FHE)
# Απαιτεί προσεκτική ρύθμιση για να αντέξει τους πολλαπλασιασμούς
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys() # Απαραίτητο για πολλαπλασιασμούς

# 2. Κρυπτογράφηση δύο πινάκων (Vectors)
vector1 = [10.5, 20.2, 30.0]
vector2 = [2.0,  3.0,  4.0]

print(f"Αρχικός Πίνακας 1: {vector1}")
print(f"Αρχικός Πίνακας 2: {vector2}")

enc_v1 = ts.ckks_vector(context, vector1)
enc_v2 = ts.ckks_vector(context, vector2)

# 3. ΠΟΛΛΑΠΛΑΣΙΑΣΜΟΣ ΚΡΥΠΤΟΓΡΑΦΗΜΕΝΩΝ ΔΕΔΟΜΕΝΩΝ
print("\nΠολλαπλασιάζουμε τα Ciphertexts στο Cloud...")
enc_result = enc_v1 * enc_v2

# 4. Αποκρυπτογράφηση
dec_result = enc_result.decrypt()

# Το αποτέλεσμα πρέπει να είναι [21.0, 60.6, 120.0]
# Θα δείτε μικρές αποκλίσεις (π.χ. 21.000000001) λόγω της προσέγγισης του CKKS!
print(f"Αποκρυπτογραφημένο Αποτέλεσμα: {[round(x, 10) for x in dec_result]}")

=== CKKS ===
Αρχικός Πίνακας 1: [10.5, 20.2, 30.0]
Αρχικός Πίνακας 2: [2.0, 3.0, 4.0]

Πολλαπλασιάζουμε τα Ciphertexts στο Cloud...
Αποκρυπτογραφημένο Αποτέλεσμα: [21.0000028094, 60.6000081237, 120.0000160892]


In [7]:
import time
import random
import tenseal as ts
from phe import paillier

NUM_SALARIES = 100
# Φτιάχνουμε τυχαίους μισθούς
salaries_float = [random.uniform(1000.0, 5000.0) for _ in range(NUM_SALARIES)]
# Για BFV/BGV χρειαζόμαστε ακεραίους
salaries_int = [int(x * 100) for x in salaries_float]

print(f"=== BENCHMARK: {NUM_SALARIES} μισθοί ===\n")

def benchmark_paillier():
    t0 = time.perf_counter()
    pub, priv = paillier.generate_paillier_keypair()
    t_key = time.perf_counter() - t0

    t0 = time.perf_counter()
    enc = [pub.encrypt(x) for x in salaries_float]
    t_enc = time.perf_counter() - t0

    t0 = time.perf_counter()
    enc_sum = sum(enc)
    t_calc = time.perf_counter() - t0

    t0 = time.perf_counter()
    res = priv.decrypt(enc_sum) / NUM_SALARIES
    t_dec = time.perf_counter() - t0
    return t_key, t_enc, t_calc, t_dec, res


def benchmark_tenseal(scheme_type, data):
    poly_mod = 8192

    if scheme_type == ts.SCHEME_TYPE.CKKS:
        context = ts.context(scheme_type, poly_mod, coeff_mod_bit_sizes=[60, 40, 40, 60])
        context.global_scale = 2**40
    else: # BFV
        # ΔΙΟΡΘΩΣΗ: Βάζουμε ένα τεράστιο plain_modulus (536,903,681)
        # για να χωρέσει το άθροισμα των χωρίς να κάνει overflow (modulo wrap-around).
        context = ts.context(scheme_type, poly_mod, plain_modulus=536903681)

    context.generate_galois_keys()

    t0 = time.perf_counter()
    t_key = time.perf_counter() - t0

    t0 = time.perf_counter()
    if scheme_type == ts.SCHEME_TYPE.CKKS:
        enc_vec = ts.ckks_vector(context, data)
    else:
        enc_vec = ts.bfv_vector(context, data)
    t_enc = time.perf_counter() - t0

    t0 = time.perf_counter()
    enc_sum = enc_vec.sum()

    if scheme_type == ts.SCHEME_TYPE.CKKS:
        enc_avg = enc_sum * (1.0 / NUM_SALARIES)
    else:
        # Το BFV δεν υποστηρίζει διαίρεση με δεκαδικούς!
        # Κρατάμε το άθροισμα κρυπτογραφημένο και θα κάνουμε τη διαίρεση ΜΕΤΑ την αποκρυπτογράφηση.
        enc_avg = enc_sum
    t_calc = time.perf_counter() - t0

    t0 = time.perf_counter()
    dec = enc_avg.decrypt()[0]

    if scheme_type != ts.SCHEME_TYPE.CKKS:
        # Επαναφορά: Διαιρούμε με το 100 (που βάλαμε αρχικά) ΚΑΙ με το NUM_SALARIES για τον μέσο όρο
        res = (dec / 100.0) / NUM_SALARIES
    else:
        res = dec
    t_dec = time.perf_counter() - t0

    return t_key, t_enc, t_calc, t_dec, res

# Εκτέλεση μετρήσεων
results = {
    "Paillier (PHE)": benchmark_paillier(),
    "CKKS (FHE Float)": benchmark_tenseal(ts.SCHEME_TYPE.CKKS, salaries_float),
    "BFV (FHE Int)": benchmark_tenseal(ts.SCHEME_TYPE.BFV, salaries_int),
}

# Εκτύπωση αποτελεσμάτων
print(f"{'Σχήμα':<20} | {'KeyGen':<8} | {'Encrypt':<8} | {'Calc':<8} | {'Decrypt':<8} | {'Result'}")
print("-" * 85)
for name, (tk, te, tc, td, res) in results.items():
    print(f"{name:<20} | {tk:.8f}s  | {te:.8f}s  | {tc:.8f}s  | {td:.8f}s  | {res:.8f}€")

=== BENCHMARK: 100 μισθοί ===

Σχήμα                | KeyGen   | Encrypt  | Calc     | Decrypt  | Result
-------------------------------------------------------------------------------------
Paillier (PHE)       | 12.17359796s  | 36.65626421s  | 0.02053396s  | 0.09734822s  | 2908.96991956€
CKKS (FHE Float)     | 0.00000110s  | 0.00746865s  | 0.06190709s  | 0.00216759s  | 2908.97030974€
BFV (FHE Int)        | 0.00000122s  | 0.00628866s  | 0.10691393s  | 0.00263863s  | 2908.96490000€
